In [ ]:
# If not installed yet:
# !pip install --upgrade openai

from openai import OpenAI
import os


In [ ]:
# === GPT CONFIG ===

client = OpenAI()  # uses OPENAI_API_KEY from env

GPT_MODEL_NAME = "gpt-4.1-mini"  # or "gpt-4.1", "gpt-5.1" etc.


In [ ]:
import re

def call_model(prompt: str) -> str:
    """
    Call GPT (e.g. gpt-4.1-mini) with the given prompt and
    return a cleaned SQL string.
    """
    # We wrap your prompt in a system message that forces: "only SQL"
    system_msg = (
        "You are a text-to-SQL model for MySQL.\n"
        "You will receive a prompt that includes a schema and a question.\n"
        "Return ONLY a single SQL query, no explanation, no markdown, no backticks."
    )

    full_input = [
        {
            "role": "system",
            "content": system_msg,
        },
        {
            "role": "user",
            "content": prompt,
        },
    ]

    # Use Responses API (latest recommended)
    resp = client.responses.create(
        model=GPT_MODEL_NAME,
        input=full_input,
        max_output_tokens=512,
    )

    # Simplest: get output as text
    raw_text = resp.output_text.strip()

    # ---- Extract SQL (SELECT/WITH) ----
    # In case GPT still adds some chatter, be defensive:
    sql_match = re.search(r"(SELECT|WITH)\s+.*", raw_text, re.IGNORECASE | re.DOTALL)
    if sql_match:
        sql = sql_match.group(0).strip()
    else:
        sql = raw_text  # fallback

    # Strip markdown fences/backticks if any
    sql = sql.replace("```sql", "").replace("```", "").strip()

    # Ensure trailing semicolon isn't mandatory, but ok if present
    return sql


In [ ]:
# === Cell: Evaluate one prompting technique (uses new call_model) ===

def evaluate_technique(technique: str, save_json: bool = True):
    print("\n" + "="*60)
    print(f"Evaluating technique: {technique}")
    print("="*60)

    schema = schema_snippet
    results = []

    # Few-shot examples: first 3
    few_shot_examples = [
        {"question": t["question"], "sql": t["gold_sql"]}
        for t in TEST_CASES[:3]
    ]

    for i, case in enumerate(TEST_CASES, 1):
        question = case["question"]
        gold_sql = case["gold_sql"]

        print(f"\n[{i}/{NUM_TEST_QUESTIONS}] Q: {question}")

        # build prompt for this technique (FS / CoT / LtM / EG)
        prompt = build_prompt_for_technique(
            technique=technique,
            schema=schema,
            question=question,
            examples=few_shot_examples,
            model_name=GPT_MODEL_NAME,  # <= just to help prompt formatting
        )

        # ----- call GPT -----
        try:
            model_sql = call_model(prompt)
            print("MODEL SQL:", model_sql)
        except Exception as e:
            print("Model call failed:", e)
            results.append({
                "technique": technique,
                "question": question,
                "model_sql": None,
                "gold_sql": gold_sql,
                "exec_success": False,
                "semantic_success": False,
                "exec_error": str(e),
                "rows": None,
            })
            continue

        # ----- execute gold + model SQL on MySQL -----
        gold_exec_success, gold_rows, gold_error   = run_sql(gold_sql)
        model_exec_success, model_rows, model_error = run_sql(model_sql)

        if not gold_exec_success:
            print("!! Gold SQL failed !!", gold_error)

        # ----- semantic comparison -----
        if gold_exec_success and model_exec_success:
            gold_norm  = normalize_rows(gold_rows)
            model_norm = normalize_rows(model_rows)
            semantic_success = (gold_norm == model_norm)
        else:
            semantic_success = False

        results.append({
            "technique": technique,
            "question": question,
            "model_sql": model_sql,
            "gold_sql": gold_sql,
            "exec_success": model_exec_success,
            "semantic_success": semantic_success,
            "exec_error": model_error,
            "rows": len(model_rows) if model_rows else 0,
        })

        print("Exec success:", model_exec_success,
              "| Semantic match:", semantic_success,
              "| Rows:", len(model_rows) if model_rows else 0)
        if model_error:
            print("Error:", model_error)

    # ----- metrics -----
    total = len(results)
    exec_successes = sum(1 for r in results if r["exec_success"])
    sem_successes  = sum(1 for r in results if r["semantic_success"])

    exec_acc = exec_successes / total * 100 if total > 0 else 0.0
    sem_acc  = sem_successes  / total * 100 if total > 0 else 0.0

    print("\n" + "-"*60)
    print(f"Technique: {technique}")
    print(f"Total: {total}")
    print(f"Execution Accuracy: {exec_acc:.2f}%  ({exec_successes}/{total})")
    print(f"Semantic Accuracy:  {sem_acc:.2f}%  ({sem_successes}/{total})")
    print("-"*60)

    if save_json:
        out_path = SCRIPT_DIR / f"adult_model_test_results_{technique}_gpt.json"
        with open(out_path, "w") as f:
            json.dump(results, f, indent=2)
        print(f"Saved detailed results to: {out_path}")

    return results, exec_acc, sem_acc


In [ ]:
PROMPTING_TECHNIQUES = ["few_shot", "cot", "ltm", "eg"]

all_results = {}
accuracy_records = []

for tech in PROMPTING_TECHNIQUES:
    res, exec_acc, sem_acc = evaluate_technique(technique=tech)
    all_results[tech] = res
    accuracy_records.append({
        "technique": tech,
        "exec_accuracy": exec_acc,
        "semantic_accuracy": sem_acc,
    })

accuracy_records
